In [1]:
import os
import math
import json
import copy
import itertools
import functools
from datetime import datetime

In [2]:
os.environ.setdefault("FINTORCH_BASE_TIME_FRAME", "60")

'60'

In [3]:
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from rich.progress import Progress

from fintorch.enum import TimeFrame
from fintorch.exchange import ONLINE_EXCHANGE
from fintorch.setting import RICH_PROGRESS_COLUMNS
from fintorch.utils.function import call_with_dict
from fintorch.utils.args import DefaultArgumentParser
from fintorch.deep.transform.label import *

In [180]:
string_args = [
    "--stop-date", "2024-06-01",
    "--symbol", "BTC-USDT",
    "--time-frame", "60",
]
args = DefaultArgumentParser.parse(args=string_args)

In [181]:
dc = ONLINE_EXCHANGE.future.data.get_data_collection(symbols=[args.symbol], time_frames=[args.time_frame])
sdf = ONLINE_EXCHANGE.future.data.get_candles_dataframe(symbol=args.symbol, time_frame=args.time_frame).copy()

sdf.insert(0, "datetime", [datetime.fromtimestamp(ts) for ts in sdf.index.to_list()])

sdf

,datetime,open,high,low,close,volume,trade
timestamp,,,,,,,
1569398400,2019-09-25 11:30:00,8325.28,8330.36,8320.85,8322.44,18.796,55
1569398460,2019-09-25 11:31:00,8329.93,8329.93,8292.16,8307.37,51.096,168
1569398520,2019-09-25 11:32:00,8308.14,8308.16,8279.93,8301.24,122.686,251
1569398580,2019-09-25 11:33:00,8296.89,8313.61,8271.86,8275.98,122.719,226
1569398640,2019-09-25 11:34:00,8276.08,8293.35,8270.93,8290.86,108.744,213
...,...,...,...,...,...,...,...
1710625380,2024-03-17 01:13:00,66227.40,66246.70,66227.30,66235.30,10.705,292
1710625440,2024-03-17 01:14:00,66378.70,66424.00,66347.80,66424.00,91.427,1736
1710625500,2024-03-17 01:15:00,66424.00,66500.00,66412.80,66415.40,133.206,3168


In [495]:
label_transform_type = functools.partial(ForwardMiddleSmaLabelTransform)
label_transform = call_with_dict(label_transform_type, args.transform_kwargs)
ldf = label_transform.transform_df(dc=dc)

ldf

,open,high,low,close,volume,trade,middle-sma,forward-middle-sma,label
timestamp,,,,,,,,,
1569398400,8325.28,8330.36,8320.85,8322.44,18.796,55,8299.266667,8321.402727,True
1569398460,8329.93,8329.93,8292.16,8307.37,51.096,168,8302.580000,8323.687273,True
1569398520,8308.14,8308.16,8279.93,8301.24,122.686,251,8308.375000,8327.944545,True
1569398580,8296.89,8313.61,8271.86,8275.98,122.719,226,8312.038889,8331.920909,True
1569398640,8276.08,8293.35,8270.93,8290.86,108.744,213,8317.826000,8339.224545,True
...,...,...,...,...,...,...,...,...,...
1710625380,66227.40,66246.70,66227.30,66235.30,10.705,292,66429.270000,66392.500000,False
1710625440,66378.70,66424.00,66347.80,66424.00,91.427,1736,66447.355556,66392.500000,False
1710625500,66424.00,66500.00,66412.80,66415.40,133.206,3168,66453.587500,66392.500000,False


In [502]:
LENGTH = 5

# indicators
df = sdf.copy()
df["tr"] = df.high / df.low - 1
df["roc"] = df.close / df.open - 1
df["atr"] = df.tr.rolling(window=LENGTH).mean()
df["aroc"] = np.abs(df.roc).rolling(window=LENGTH).mean()

# indicators
df["csroc"] = np.cumsum(df.roc)
df["bbm"] = df.csroc.rolling(window=LENGTH).mean()
df["bbs"] = df.csroc.rolling(window=LENGTH).std()
df["bbl"] = df.bbm - df.bbs
df["bbh"] = df.bbm + df.bbs

# farctals
df["bmax"] = df.csroc.rolling(window=LENGTH).max()
df["bmin"] = df.csroc.rolling(window=LENGTH).min()
df["distmax"] = np.abs(df.bmax - df.csroc)
df["distmin"] = np.abs(df.bmin - df.csroc)
df["farmax"] = df.distmax.mean() + df.distmax.std() <= df.distmax
df["farmin"] = df.distmin.mean() + df.distmin.std() <= df.distmin
df["nearmax"] = df.distmax <= df.distmax.mean() + df.distmax.std()
df["nearmin"] = df.distmin <= df.distmin.mean() + df.distmin.std()

# buy signals                              
# df["buy"] = True
# df["buy"] = df.bbs < df.atr
# df["buy"] = df.nearmax
# df["buy"] = df.bbm < df.csroc
# df["buy"] = df.bbh < df.csroc
df["buy"] = 1 == np.where(0.7 < np.random.rand(len(df)), (ldf.label + 1) % 2,  ldf.label)

df.buy.sum() / len(df)

0.5008777341034961

In [503]:
TPR = 3
SLR = 1

tdf = df.iloc[-100000:]

#######################
###### backtest #######
#######################
profits_list, bars_list = [], []

entry_price = None
for index in tqdm(range(len(tdf))):
    if entry_price is None and tdf.buy.iloc[index]:
        entry_price = tdf.close.iloc[index]
        stop_loss = tdf.aroc.iloc[index] * SLR
        take_profit = tdf.aroc.iloc[index] * TPR
    
        stop_loss_price = round(entry_price * (1 - stop_loss - 0.0002 * 2), 2)
        take_profit_price = round(entry_price * (1 + take_profit + 0.0002 * 2), 2)
            
    elif entry_price is not None:
        profit = None
        if tdf.low.iloc[index] <= stop_loss_price:
            profit = -stop_loss
        elif take_profit_price <= tdf.high.iloc[index]:
            profit = take_profit

        if profit is not None:
            profits_list.append(profit)
            entry_price = None

100%|██████████| 100000/100000 [00:00<00:00, 112893.47it/s]


In [504]:
LEVERAGE = 10

profits = np.array(profits_list) * LEVERAGE

days = len(tdf) * TimeFrame.MIN1 // TimeFrame.DAY1

trades = len(profits)
daily_trades = len(profits) // days

net_profit = profits.sum()
compound_profit = (1 + profits).prod() - 1
daily_net_profit = net_profit / days
daily_compound_profit = (compound_profit + 1) ** (1 / days) - 1

wining_ratio = (0 < profits).sum() / len(profits) * 100

# print statements
print("{:<32}{}".format("Days", days))
print()
print("{:<32}{}".format("Trades", trades))
print("{:<32}{}".format("Daily trades", daily_trades))
print()
print("{:<32}{:.2f}".format("Net profit", net_profit))
print("{:<32}{:.2f}".format("Compound profit", compound_profit))
print("{:<32}{:.2f}".format("Daily net profit", daily_net_profit))
print("{:<32}{:.2f}".format("Daily compound profit", daily_compound_profit))
print()
print("{:<32}{:.2f}%".format("Wining ratio", wining_ratio))


Days                            69

Trades                          9857
Daily trades                    142

Net profit                      23.11
Compound profit                 3123684299.14
Daily net profit                0.33
Daily compound profit           0.37

Wining ratio                    36.51%
